# optimizer-repr-string — ex2: multi-param-group __repr__ matching PyTorch's actual format

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `optimizer-repr-string`. Running the final beacon cell reports progress against the `Optimizer: __repr__ string` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: __repr__ string` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`optimizer-repr-string`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "optimizer-repr-string"
DD_SUBTOPIC = "Optimizer: __repr__ string"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Multi-param-group `__repr__` — PyTorch's actual format

Ex1 built a single-group repr. Real `torch.optim.SGD` supports MULTIPLE param groups (different lr/momentum per group — common for finetuning with head-vs-backbone lrs). The repr format PyTorch uses:

```
SGD (
Parameter Group 0
    lr: 0.001
    momentum: 0.9
Parameter Group 1
    lr: 0.0001
    momentum: 0.0
)
```

**Keys are sorted alphabetically within each group.** PyTorch enforces this for stable diffs across versions.

**Header = class name + space + `(`.** Footer = `)` on its own line. Each `Parameter Group N` header is followed by `    key: value` lines (4-space indent).

**Why this matters.** When a finetune script prints `optim`, the instructor needs to see `lr=0.001` for the head and `lr=0.0001` for the backbone in ONE glance — a flat single-group repr hides the bug.

### Exercise 2 — multi-param-group __repr__ matching PyTorch's actual format

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply PyTorch's optimizer-repr format — header `<Name> (`, one `Parameter Group <i>` block per group with alphabetically-sorted `    key: value` lines, footer `)` — to a hand-rolled SGD that stores its hparams as a list of group dicts.
> Keywords: repr, param-groups, optimizer, format
> ```

**KCs targeted:** `param-group-header-format`, `alphabetical-key-ordering`

Implement `ex2_format_multigroup_sgd(param_groups)`. Given a list of param-group dicts, return PyTorch's canonical multi-group repr string.

Inputs:
- `param_groups`: `list[dict]`. Each dict contains hparam keys (e.g. `'lr'`, `'momentum'`, `'weight_decay'`, ...). Each dict MAY include a `'params'` key — IGNORE it for the repr (PyTorch does too).

Output: single `str` with this EXACT shape (note: trailing `)` on its own line, NO trailing newline):

```
SGD (
Parameter Group 0
    lr: 0.001
    momentum: 0.9
Parameter Group 1
    lr: 0.0001
    momentum: 0.0
    weight_decay: 0.0005
)
```

Rules:
1. Header line: literal `'SGD ('`.
2. Per group: `'Parameter Group <i>'` line (zero-indexed), then one `'    <key>: <value>'` line per hparam, with keys sorted ALPHABETICALLY, excluding `'params'`.
3. Values rendered with `str(value)` — float `0.001` becomes `'0.001'`, int `64` becomes `'64'`, etc.
4. Footer line: literal `')'`.
5. Lines joined with `'\n'`; NO trailing newline at the end.

In [ ]:
def ex2_format_multigroup_sgd(param_groups):
    lines = ['SGD (']
    for i, group in enumerate(param_groups):
        lines.append(f'Parameter Group {i}')
        keys = sorted(k for k in group.keys() if k != 'params')
        for k in keys:
            lines.append(f'    {k}: {group[k]}')
    lines.append(')')
    return '\n'.join(lines)


<details><summary>Solution</summary>

```python
def ex2_format_multigroup_sgd(param_groups):
    lines = ['SGD (']
    for i, group in enumerate(param_groups):
        lines.append(f'Parameter Group {i}')
        keys = sorted(k for k in group.keys() if k != 'params')
        for k in keys:
            lines.append(f'    {k}: {group[k]}')
    lines.append(')')
    return '\n'.join(lines)
```

**Sort keys per group, not globally.** Different groups can have different hparam sets (e.g. group 1 has `weight_decay`, group 0 doesn't). Sort each group's keys independently.

**Exclude `'params'` before sorting.** The params list is one or more `nn.Parameter` objects — printing them dumps tensor reprs into the optimizer repr and makes it useless. PyTorch's own `Optimizer.__repr__` does this exclusion at the source.

**`'\n'.join(lines)` not `'\n'.join(lines) + '\n'`.** PyTorch's actual repr has no trailing newline — that's a `print()` choice, not part of the string. Adding one breaks string-equality tests.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()